# Datasets generation

In [3]:
import myutils as utils
import os

working_dir = os.getenv('SCRATCHDIR_USER')
if working_dir is None:
    working_dir = os.getenv("HOME")
else:
    working_dir = f"/work/{working_dir.split("/")[-1]}/shortcut-loc-exp"
# utils.datasets.gen_cifar_10_mnist(root=f"{working_dir}/datasets", split="test", match_seed=1, s_size=10, s_a=1/4)
# utils.datasets.gen_waterbirds(root=f"{working_dir}/datasets", match_seed=2)

# Evaluation data analysis preparation

In [1]:
import math
import os
import pickle
import numpy as np
import scipy.stats as sps


working_dir = os.getenv('SCRATCHDIR_USER')
if working_dir is None:
    working_dir = os.getenv("HOME")
checkpoint_path = f"{working_dir}/checkpoints"

In [2]:
def to_percent(x: np.ndarray, precision: int = 1) -> np.ndarray:
    """Round the number for percent reporting"""
    scale = 10**(precision+2)
    return np.ceil(scale*x)/(scale/100)


def get_stats(
        val_contrib: np.ndarray, bad_points: np.ndarray
        ) -> tuple[np.ndarray, np.ndarray]:
    """Get mean and standard error of individual contributions"""
    means = []
    stds = []
    for i in range(len(val_contrib)):
        inv_n = (~bad_points[i]).sum() - 1
        if inv_n >= 0:
            means.append(val_contrib[i, ~bad_points[i]].mean())
        else:
            means.append(np.nan)
        if inv_n > 0:
            inv_n = (inv_n)**(-0.5)
            stds.append(val_contrib[i, ~bad_points[i]].std()*inv_n)
        else:
            stds.append(np.nan)
    return np.array(means), np.array(stds)


def get_contrib(
        val_loss: np.ndarray, s_val_loss: np.ndarray, split_name: str
        ) -> tuple[np.ndarray, np.ndarray]:
    """Get individual layers' contributions"""
    c_ind, s_ind = 0, -1
    if split_name[0] == "c":
        c_ind, s_ind = -1, 0
    val_err = val_loss[1:-1] - val_loss[s_ind]
    s_val_err = s_val_loss[1:-1] - s_val_loss[c_ind]
    bad_points = (val_err > 0) & (s_val_err > 0)

    val_contrib = (val_loss[:1] - val_loss[1:-1]) / (val_loss[:1] - val_loss[-1:])
    return get_stats(val_contrib, bad_points)


def get_sum_contrib(
        val_loss: np.ndarray, s_val_loss: np.ndarray, split_name: str
        ) -> tuple[np.ndarray, np.ndarray]:
    """Get individual layers' contributions"""
    c_ind, s_ind = 0, -1
    if split_name[0] == "c":
        c_ind, s_ind = -1, 0
    val_err = val_loss[1:-1] - val_loss[s_ind]
    s_val_err = s_val_loss[1:-1] - s_val_loss[c_ind]
    bad_points = (val_err > 0) & (s_val_err > 0)

    val_contrib = (val_loss[:1] - val_loss[1:-1]) / (val_loss[:1] - val_loss[-1:])
    sum_contrib = val_contrib.sum(0)[None, :]
    bad_points = bad_points.any(0)[None, :]
    return get_stats(sum_contrib, bad_points)


NAMES = {"v11": "VGG-11", "r18": "ResNet-18", "dtt": "DeiT-Ti", "cnt": "ConvNeXt-Ti"}
DATA_NAMES = {
    "c10m-w": "CIFAR-10 (weak)", "c10m-s": "CIFAR-10 (strong)",
    "wb": "Waterbirds", "ca": "CelebA"}
inv_n = (5-1)**(-0.5)

# Check divergences

In [ ]:
train_type = "f"
samples_list = ["c10m-s", "c10m-w", "ca", "wb"]

bad_models = 0
all_models = 0
for samples_name in samples_list:
# for samples_name in ["c10m-s"]:
    for freq in ["r", "c"]:
    # for freq in ["c"]:
        for model_name in ["dtt", "r18", "v11", "cnt"]:
            if train_type == "s" and (
                (model_name=="dtt" and samples_name=="wb")\
                or (model_name=="cnt")):
                continue
            for split_name in ["ch", "sh", "cl", "sl"]: 
            # for split_name in ["ch", "sh"]: 
            # for split_name in ["cl", "sl"]: 
                with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
                    val_losses = pickle.load(f)
                with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
                    s_val_losses = pickle.load(f) 
                for optimizer_name in ["adamw", "sgd"]:
                # for optimizer_name in ["adamw"]:
                    val_loss = val_losses[
                        f"{samples_name}-{freq}-{model_name}-"\
                        f"{train_type}-{optimizer_name}"]
                    s_val_loss = s_val_losses[
                        f"{samples_name}-{freq}-{model_name}-"\
                        f"{train_type}-{optimizer_name}"]
                    c_ind, s_ind = 0, -1
                    if split_name[0] == "c":
                        c_ind, s_ind = -1, 0
                    val_contrib = val_loss[1:-1] - val_loss[s_ind]
                    s_val_err = s_val_loss[1:-1] - s_val_loss[c_ind]
                    bad_points = (val_contrib > 0) & (s_val_err > 0)
                    bad_models += bad_points.sum()
                    all_models += bad_points.size
                    if np.any(bad_points):
                        print(
                            f"{split_name} {samples_name:>6} {freq} {model_name}",
                            f"{train_type} {optimizer_name:>5}",
                            bad_points.sum(1), bad_points.sum(), bad_points.size)
print(bad_models, all_models)

# Contributions of individual layers

In [ ]:
train_type = "f"
samples_name = "c10m-s"
freq = "c"

for split_name in ["cl", "sl"]:
    with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
        val_losses = pickle.load(f)
    with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
        s_val_losses = pickle.load(f)

    for i, model_name in enumerate(["dtt", "r18", "v11", "cnt"]):
        val_loss = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-adamw"]
        s_val_loss = s_val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-adamw"]
        val_mean, val_std = get_contrib(val_loss, s_val_loss, split_name)
        print(
            f"{NAMES[model_name]} &",
            " & ".join(map(
                lambda x: fr"${x:3.1f}\%$", to_percent(val_mean))),
            "\\\\")
        print(
            "&",
            " & ".join(map(
                lambda x: fr"$({x:3.1f}\%)$", to_percent(val_std))),
            "\\\\")


for split_name in ["cl", "sl"]:
    with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
        val_losses = pickle.load(f)
    with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
        s_val_losses = pickle.load(f)

    for i, model_name in enumerate(["dtt", "r18", "v11", "cnt"]):
        val_loss = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-adamw"]
        s_val_loss = s_val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-adamw"]
        val_mean, val_std = get_sum_contrib(val_loss, s_val_loss, split_name)
        print(f"{NAMES[model_name]} t-value:", (val_mean - 1) / val_std)

print("One-sided 5% critical value:", sps.t.ppf(0.95, 5-1))
print("Two-sided 5% critical value:", sps.t.ppf(0.975, 5-1))

# Fine-tuned models

## Error rates

In [ ]:
split_name = "sl"
with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
    val_losses = pickle.load(f)
with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
    s_val_losses = pickle.load(f)

train_type = "f"
optim_name = "adamw"
for freq in ["r", "c"]:
    err_rates = [[] for _ in range(len(NAMES))]
    std_rates = [[] for _ in range(len(NAMES))]
    for i, model_name in enumerate(["dtt", "r18", "v11", "cnt"]):
        for data_name in ["c10m-w", "c10m-s", "wb", "ca"]:
            val_contrib = val_losses[f"{data_name}-{freq}-{model_name}-{train_type}-{optim_name}"]
            err_rates[i].append(val_contrib[0].mean())
            err_rates[i].append(val_contrib[-1].mean())
            std_rates[i].append(val_contrib[0].std()*inv_n)
            std_rates[i].append(val_contrib[-1].std()*inv_n)

    for i, model_name in enumerate(["dtt", "r18", "v11", "cnt"]):
        print(
            f"{NAMES[model_name]} &",
            " & ".join(map(
                lambda x: fr"${x:3.1f}\%$", to_percent(np.array(err_rates[i])))),
            "\\\\")
        print(
            "&",
            " & ".join(map(
                lambda x: fr"$({x:3.1f}\%)$", to_percent(np.array(std_rates[i])))),
            "\\\\")

## Errors on absoulutely skewed dataset

In [ ]:
split_name = "cl"
with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
    val_losses = pickle.load(f)
with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
    s_val_losses = pickle.load(f)

samples_name = "ca"
freq = "c"
for i, model_name in enumerate(["dtt", "r18", "v11", "cnt"]):
    val_contrib = s_val_losses[f"{samples_name}-{freq}-{model_name}-f-adamw"]
    print(
        f"{NAMES[model_name]} &",
        " & ".join(map(
            lambda x: fr"${x:3.2f}\%$", to_percent(val_contrib.mean(1), 2))),
        "\\\\")
    print(
        "&",
        " & ".join(map(
            lambda x: fr"$({x:3.2f}\%)$", to_percent(val_contrib.std(1)*inv_n, 2))),
        "\\\\")

## Explanatory factors

In [ ]:
train_type = "f"

split_name = "sh"
with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
    val_losses = pickle.load(f)
with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
    s_val_losses = pickle.load(f)

models = ["dtt", "r18", "v11", "cnt"]

print("Forgetting" if split_name[0] == "s" else "Encoding")

shares = []
for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
    for freq in ["r", "c"]:
        for model_name in models:
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
shares = np.stack(shares)
all_var = ((shares - shares.mean(0))**2).sum()
all_shares = shares

sample_shares = []
samples_vars = []
for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
    shares = []
    for freq in ["r", "c"]:
        for model_name in models:
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
    shares = np.stack(shares)
    samples_vars.append(((shares - shares.mean(0))**2).sum())
    sample_shares.append(shares.mean(0))
sample_shares = np.stack(sample_shares)

freq_shares = []
freq_vars = []
for freq in ["r", "c"]:
    shares = []
    for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
        for model_name in models:
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
    shares = np.stack(shares)
    freq_vars.append(((shares - shares.mean(0))**2).sum())
    freq_shares.append(shares.mean(0))
freq_shares = np.stack(freq_shares)


model_shares = []
model_vars = []
for model_name in models:
    shares = []
    for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
        for freq in ["r", "c"]:
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
    shares = np.stack(shares)
    model_vars.append(((shares - shares.mean(0))**2).sum())
    model_shares.append(shares.mean(0))
model_shares = np.stack(model_shares)


optim_shares = []
optim_vars = []
for optimizer_name in ["adamw", "sgd"]:
    shares = []
    for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
        for freq in ["r", "c"]:
            for model_name in models:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
    shares = np.stack(shares)
    optim_vars.append(((shares - shares.mean(0))**2).sum())
    optim_shares.append(shares.mean(0))
optim_shares = np.stack(optim_shares)

# print(avg_shares, sample_shares, model_shares, optim_shares, sep="\n")
print(f"Samples var: {1 - sum(samples_vars)/all_var:3.1%}")
print(f"Freq var: {1 - sum(freq_vars)/all_var:3.1%}")
print(f"Model var: {1 - sum(model_vars)/all_var:3.1%}")
print(f"Optimizer var: {1 - sum(optim_vars)/all_var:3.1%}")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'

avg_shares = []
for split_name in ["sh", "ch"]:
    with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
        val_losses = pickle.load(f)
    with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
        s_val_losses = pickle.load(f)

    models = ["dtt", "r18", "v11", "cnt"]

    print("Forgetting" if split_name[0] == "s" else "Encoding")

    shares = []
    for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
        for freq in ["r", "c"]:
            for model_name in models:
                for optimizer_name in ["adamw", "sgd"]:
                    share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                    shares.append(share.mean(1))
    avg_shares.append(np.stack(shares).mean(0))

x_tick = tuple(f"Block {i}" for i in range(6))
x = np.arange(6)
plt.figure(figsize=(3.3, 2.4), dpi=300)
plt.plot(x, avg_shares[0], marker="o", label="Forgetting", linewidth=0.5, markersize=4)
plt.plot(x, avg_shares[1], marker="o", label="Encoding", linewidth=0.5, markersize=4)
plt.xticks(x, x_tick)
plt.tick_params(
    axis='both', labelsize=7, length=2, width=0.5)
plt.legend(fontsize=6)
plt.setp(plt.gca().spines.values(), linewidth=0.6)
# plt.title("Average relative encoding and forgetting rates", fontsize=7)
plt.gcf()
plt.savefig("fig2-avg-enc-fgt.pdf", bbox_inches='tight', pad_inches=0)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'

avg_shares = []

for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
    split_name = "ch"
    with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
        val_losses = pickle.load(f)
    with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
        s_val_losses = pickle.load(f)

    models = ["dtt", "r18", "v11", "cnt"]

    print("Forgetting" if split_name[0] == "s" else "Encoding")

    shares = []
    for freq in ["r", "c"]:
        for model_name in models:
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
    avg_shares.append(np.stack(shares).mean(0))


for model_name in ["dtt", "r18", "v11", "cnt"]:
    split_name = "ch"
    with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
        val_losses = pickle.load(f)
    with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
        s_val_losses = pickle.load(f)

    print("Forgetting" if split_name[0] == "s" else "Encoding")

    shares = []
    for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
        for freq in ["r", "c"]:
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
    avg_shares.append(np.stack(shares).mean(0))


for freq in ["r", "c"]:
    split_name = "sh"
    with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
        val_losses = pickle.load(f)
    with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
        s_val_losses = pickle.load(f)

    models = ["dtt", "r18", "v11", "cnt"]

    print("Forgetting" if split_name[0] == "s" else "Encoding")

    shares = []
    for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
        for model_name in models:
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
    avg_shares.append(np.stack(shares).mean(0))


for optimizer_name in ["adamw", "sgd"]:
    split_name = "sh"
    with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
        val_losses = pickle.load(f)
    with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
        s_val_losses = pickle.load(f)

    models = ["dtt", "r18", "v11", "cnt"]

    print("Forgetting" if split_name[0] == "s" else "Encoding")

    shares = []
    for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
        for freq in ["r", "c"]:
            for model_name in models:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
    avg_shares.append(np.stack(shares).mean(0))

x_tick = tuple(f"Block {i}" for i in range(6))
x = np.arange(6)
fig, ax = plt.subplots(2, 2, constrained_layout=True)
fig.set_size_inches(6.6, 4.8)
ax[0][0].set_title("Average encoding", fontsize=7)
ax[0][0].plot(x, avg_shares[0], marker="o", label="CIFAR-10 (strong)", linewidth=0.5, markersize=4)
ax[0][0].plot(x, avg_shares[1], marker="o", label="CIFAR-10 (weak)", linewidth=0.5, markersize=4)
ax[0][0].plot(x, avg_shares[2], marker="o", label="CelebA", linewidth=0.5, markersize=4)
ax[0][0].plot(x, avg_shares[3], marker="o", label="Waterbirds", linewidth=0.5, markersize=4)
ax[0][1].set_title("Average encoding", fontsize=7)
ax[0][1].plot(x, avg_shares[4], marker="o", label="DeiT-Ti", linewidth=0.5, markersize=4)
ax[0][1].plot(x, avg_shares[5], marker="o", label="ResNet-18", linewidth=0.5, markersize=4)
ax[0][1].plot(x, avg_shares[6], marker="o", label="VGG-11", linewidth=0.5, markersize=4)
ax[0][1].plot(x, avg_shares[7], marker="o", label="ConvNeXt-T", linewidth=0.5, markersize=4)
ax[1][0].set_title("Average forgetting", fontsize=7)
ax[1][0].plot(x, avg_shares[8], marker="o", label="Common", linewidth=0.5, markersize=4)
ax[1][0].plot(x, avg_shares[9], marker="o", label="Near-Universal", linewidth=0.5, markersize=4)
ax[1][1].set_title("Average forgetting", fontsize=7)
ax[1][1].plot(x, avg_shares[10], marker="o", label="AdamW", linewidth=0.5, markersize=4)
ax[1][1].plot(x, avg_shares[11], marker="o", label="SGD", linewidth=0.5, markersize=4)

for i in range(2):
    for j in range(2):
        ax[i][j].set_xticks(x, x_tick)
        ax[i][j].tick_params(
            axis='both', labelsize=7, length=2, width=0.5)
        ax[i][j].legend(fontsize=6)
        plt.setp(ax[i][j].spines.values(), linewidth=0.6)
# plt.title("Average relative encoding and forgetting rates", fontsize=7)
plt.gcf()
plt.savefig("fig3-enc-fgt-factors.pdf", bbox_inches='tight', pad_inches=0)

In [ ]:
split_name = "ch"
with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
    val_losses = pickle.load(f)
with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
    s_val_losses = pickle.load(f)

models = ["dtt", "r18", "v11", "cnt"]
print("Forgetting" if split_name[0] == "s" else "Encoding")

shares = []
for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
    for freq in ["r", "c"]:
        for model_name in models:
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
shares = np.stack(shares)
all_var = ((shares - shares.mean(0))**2).sum()
all_shares = shares

sample_model_shares = []
sample_model_vars = []
for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
    for model_name in models:
        shares = []
        for freq in ["r", "c"]:
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
        shares = np.stack(shares)
        sample_model_vars.append(((shares - shares.mean(0))**2).sum())
        sample_model_shares.append(shares.mean(0))
sample_model_shares = np.stack(sample_model_shares)

print(f"Samples and model var: {1 - sum(sample_model_vars)/all_var:3.1%}")

split_name = "sh"
with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
    val_losses = pickle.load(f)
with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
    s_val_losses = pickle.load(f)

print("Forgetting" if split_name[0] == "s" else "Encoding")

shares = []
for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
    for freq in ["r", "c"]:
        for model_name in models:
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
shares = np.stack(shares)
all_var = ((shares - shares.mean(0))**2).sum()
all_shares = shares

sample_model_shares = []
sample_model_vars = []
for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
    for model_name in models:
        shares = []
        for freq in ["r", "c"]:
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
        shares = np.stack(shares)
        sample_model_vars.append(((shares - shares.mean(0))**2).sum())
        sample_model_shares.append(shares.mean(0))
sample_model_shares = np.stack(sample_model_shares)

freq_optim_shares = []
freq_optim_vars = []
for freq in ["r", "c"]:
    for optimizer_name in ["adamw", "sgd"]:
        shares = []
        for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
            for model_name in models:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
        shares = np.stack(shares)
        freq_optim_vars.append(((shares - shares.mean(0))**2).sum())
        freq_optim_shares.append(shares.mean(0))
freq_optim_shares = np.stack(freq_optim_shares)

print(f"Samples and model var: {1 - sum(sample_model_vars)/all_var:3.1%}")
print(f"Frequency and optimizer var: {1 - sum(freq_optim_vars)/all_var:3.1%}")

## Encoding and forgetting

In [ ]:
for split_name in ["ch", "sh"]:
    with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
        val_losses = pickle.load(f)
    with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
        s_val_losses = pickle.load(f)

    for data_name in ["c10m-w", "c10m-s", "wb", "ca"]:
        for model_name in models:
            val_contrib = val_losses[f"{data_name}-c-{model_name}-f-adamw"]
            if split_name[1] == "h":
                val_contrib = (val_contrib[:-1] - val_contrib[1:]) / (val_contrib[:1] - val_contrib[-1:])
            else:
                val_contrib = (val_contrib[:1] - val_contrib[1:-1]) / (val_contrib[:1] - val_contrib[-1:])
            print(
                f"& {NAMES[model_name]} &",
                " & ".join(map(
                    lambda x: fr"${x:3.1f}\%$", to_percent(val_contrib.mean(1)))),
                "\\\\")
            print(
                "& &",
                " & ".join(map(
                    lambda x: fr"$({x:3.1f}\%)$", to_percent(val_contrib.std(1)*inv_n))),
                "\\\\")

In [ ]:
split_name = "sh"
with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
    val_losses = pickle.load(f)
with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
    s_val_losses = pickle.load(f)


OPTIM_NAME = {"sgd": "SGD", "adamw": "AdamW"}

for freq in ["r", "c"]:
    for optimizer_name in ["adamw", "sgd"]:
        val_contrib = val_losses[f"c10m-s-{freq}-r18-f-{optimizer_name}"]
        if split_name[1] == "h":
            val_contrib = (val_contrib[:-1] - val_contrib[1:]) / (val_contrib[:1] - val_contrib[-1:])
        else:
            val_contrib = (val_contrib[:1] - val_contrib[1:-1]) / (val_contrib[:1] - val_contrib[-1:])
        print(
            f"& {OPTIM_NAME[optimizer_name]} &",
            " & ".join(map(
                lambda x: fr"${x:3.1f}\%$", to_percent(val_contrib.mean(1)))),
            "\\\\")
        print(
            "& &",
            " & ".join(map(
                lambda x: fr"$({x:3.1f}\%)$", to_percent(val_contrib.std(1)*inv_n))),
            "\\\\")

# Trained from scratch models

## Error rates

In [ ]:

train_type = "s"
split_name = "sl"
with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
    val_losses = pickle.load(f)
with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
    s_val_losses = pickle.load(f)

for optim_name in ["adamw", "sgd"]:
    for freq in ["r", "c"]:
        err_rates = [[] for _ in range(3)]
        std_rates = [[] for _ in range(3)]
        for i, model_name in enumerate(["dtt", "r18", "v11"]):
            for data_name in ["c10m-w", "c10m-s", "wb", "ca"]:
                if model_name == "dtt" and data_name == "wb":
                    continue
                val_contrib = val_losses[f"{data_name}-{freq}-{model_name}-{train_type}-{optim_name}"]
                err_rates[i].append(val_contrib[0].mean())
                err_rates[i].append(val_contrib[-1].mean())
                std_rates[i].append(val_contrib[0].std()*inv_n)
                std_rates[i].append(val_contrib[-1].std()*inv_n)

        for i, model_name in enumerate(["dtt", "r18", "v11"]):
            print(
                f"{NAMES[model_name]} &",
                " & ".join(map(
                    lambda x: fr"${x:3.1f}\%$", to_percent(np.array(err_rates[i])))),
                "\\\\")
            print(
                "&",
                " & ".join(map(
                    lambda x: fr"$({x:3.1f}\%)$", to_percent(np.array(std_rates[i])))),
                "\\\\")

## Encoding and forgetting

In [ ]:
for split_name in ["ch", "sh"]:
    with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
        val_losses = pickle.load(f)
    with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
        s_val_losses = pickle.load(f)

    for data_name in ["c10m-w", "c10m-s", "wb", "ca"]:
        for model_name in ["dtt", "r18", "v11"]:
            if model_name == "dtt" and data_name == "wb":
                continue
            val_contrib = val_losses[f"{data_name}-c-{model_name}-s-adamw"]
            if split_name[1] == "h":
                val_contrib = (val_contrib[:-1] - val_contrib[1:]) / (val_contrib[:1] - val_contrib[-1:])
            else:
                val_contrib = (val_contrib[:1] - val_contrib[1:-1]) / (val_contrib[:1] - val_contrib[-1:])
            print(
                f"& {NAMES[model_name]} &",
                " & ".join(map(
                    lambda x: fr"${x:3.1f}\%$", to_percent(val_contrib.mean(1)))),
                "\\\\")
            print(
                "& &",
                " & ".join(map(
                    lambda x: fr"$({x:3.1f}\%)$", to_percent(val_contrib.std(1)*inv_n))),
                "\\\\")

In [ ]:
split_name = "sh"
with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
    val_losses = pickle.load(f)
with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
    s_val_losses = pickle.load(f)


OPTIM_NAME = {"sgd": "SGD", "adamw": "AdamW"}

for freq in ["r", "c"]:
    for optimizer_name in ["adamw", "sgd"]:
        val_contrib = val_losses[f"c10m-s-{freq}-r18-s-{optimizer_name}"]
        if split_name[1] == "h":
            val_contrib = (val_contrib[:-1] - val_contrib[1:]) / (val_contrib[:1] - val_contrib[-1:])
        else:
            val_contrib = (val_contrib[:1] - val_contrib[1:-1]) / (val_contrib[:1] - val_contrib[-1:])
        print(
            f"& {OPTIM_NAME[optimizer_name]} &",
            " & ".join(map(
                lambda x: fr"${x:3.1f}\%$", to_percent(val_contrib.mean(1)))),
            "\\\\")
        print(
            "& &",
            " & ".join(map(
                lambda x: fr"$({x:3.1f}\%)$", to_percent(val_contrib.std(1)*inv_n))),
            "\\\\")

## Explanatory factors

In [ ]:
train_type = "s"

split_name = "sh"
with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
    val_losses = pickle.load(f)
with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
    s_val_losses = pickle.load(f)

print("Forgetting" if split_name[0] == "s" else "Encoding")

shares = []
for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
    for freq in ["r", "c"]:
        for model_name in ["dtt", "r18", "v11"]:
            if model_name == "dtt" and samples_name == "wb":
                continue
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
shares = np.stack(shares)
all_var = ((shares - shares.mean(0))**2).sum()
all_shares = shares

sample_shares = []
samples_vars = []
for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
    shares = []
    for freq in ["r", "c"]:
        for model_name in ["dtt", "r18", "v11"]:
            if model_name == "dtt" and samples_name == "wb":
                continue
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
    shares = np.stack(shares)
    samples_vars.append(((shares - shares.mean(0))**2).sum())
    sample_shares.append(shares.mean(0))
sample_shares = np.stack(sample_shares)

freq_shares = []
freq_vars = []
for freq in ["r", "c"]:
    shares = []
    for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
        for model_name in ["dtt", "r18", "v11"]:
            if model_name == "dtt" and samples_name == "wb":
                continue
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
    shares = np.stack(shares)
    freq_vars.append(((shares - shares.mean(0))**2).sum())
    freq_shares.append(shares.mean(0))
freq_shares = np.stack(freq_shares)


model_shares = []
model_vars = []
for model_name in ["dtt", "r18", "v11"]:
    shares = []
    for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
        if model_name == "dtt" and samples_name == "wb":
            continue
        for freq in ["r", "c"]:
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
    shares = np.stack(shares)
    model_vars.append(((shares - shares.mean(0))**2).sum())
    model_shares.append(shares.mean(0))
model_shares = np.stack(model_shares)


optim_shares = []
optim_vars = []
for optimizer_name in ["adamw", "sgd"]:
    shares = []
    for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
        for freq in ["r", "c"]:
            for model_name in ["dtt", "r18", "v11"]:
                if model_name == "dtt" and samples_name == "wb":
                    continue
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
    shares = np.stack(shares)
    optim_vars.append(((shares - shares.mean(0))**2).sum())
    optim_shares.append(shares.mean(0))
optim_shares = np.stack(optim_shares)

# print(avg_shares, sample_shares, model_shares, optim_shares, sep="\n")
print(f"Samples var: {1 - sum(samples_vars)/all_var:3.1%}")
print(f"Freq var: {1 - sum(freq_vars)/all_var:3.1%}")
print(f"Model var: {1 - sum(model_vars)/all_var:3.1%}")
print(f"Optimizer var: {1 - sum(optim_vars)/all_var:3.1%}")

In [ ]:
train_type = "s"

split_name = "ch"
with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
    val_losses = pickle.load(f)
with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
    s_val_losses = pickle.load(f)

print("Forgetting" if split_name[0] == "s" else "Encoding")

shares = []
for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
    for freq in ["r", "c"]:
        for model_name in ["dtt", "r18", "v11"]:
            if model_name == "dtt" and samples_name == "wb":
                continue
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
shares = np.stack(shares)
all_var = ((shares - shares.mean(0))**2).sum()
all_shares = shares

sample_model_shares = []
sample_model_vars = []
for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
    for model_name in ["dtt", "r18", "v11"]:
        if model_name == "dtt" and samples_name == "wb":
            continue
        shares = []
        for freq in ["r", "c"]:
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
        shares = np.stack(shares)
        sample_model_vars.append(((shares - shares.mean(0))**2).sum())
        sample_model_shares.append(shares.mean(0))
sample_model_shares = np.stack(sample_model_shares)

print(f"Samples and model var: {1 - sum(sample_model_vars)/all_var:3.1%}")

split_name = "sh"
with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
    val_losses = pickle.load(f)
with open(f'{checkpoint_path}/{split_name}-err-s.pickle', 'rb') as f:
    s_val_losses = pickle.load(f)

print("Forgetting" if split_name[0] == "s" else "Encoding")

shares = []
for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
    for freq in ["r", "c"]:
        for model_name in ["dtt", "r18", "v11"]:
            if model_name == "dtt" and samples_name == "wb":
                continue
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
shares = np.stack(shares)
all_var = ((shares - shares.mean(0))**2).sum()
all_shares = shares

sample_model_shares = []
sample_model_vars = []
for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
    for model_name in ["dtt", "r18", "v11"]:
        if model_name == "dtt" and samples_name == "wb":
            continue
        shares = []
        for freq in ["r", "c"]:
            for optimizer_name in ["adamw", "sgd"]:
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
        shares = np.stack(shares)
        sample_model_vars.append(((shares - shares.mean(0))**2).sum())
        sample_model_shares.append(shares.mean(0))
sample_model_shares = np.stack(sample_model_shares)

freq_optim_shares = []
freq_optim_vars = []
for freq in ["r", "c"]:
    for optimizer_name in ["adamw", "sgd"]:
        shares = []
        for samples_name in ["c10m-s", "c10m-w", "ca", "wb"]:
            for model_name in ["dtt", "r18", "v11"]:
                if model_name == "dtt" and samples_name == "wb":
                    continue
                share = val_losses[f"{samples_name}-{freq}-{model_name}-{train_type}-{optimizer_name}"]
                if split_name[1] == "h":
                    share = (share[:-1] - share[1:]) / (share[:1] - share[-1:])
                else:
                    share = (share[:1] - share[1:-1]) / (share[:1] - share[-1:])
                shares.append(share.mean(1))
        shares = np.stack(shares)
        freq_optim_vars.append(((shares - shares.mean(0))**2).sum())
        freq_optim_shares.append(shares.mean(0))
freq_optim_shares = np.stack(freq_optim_shares)

print(f"Samples and model var: {1 - sum(sample_model_vars)/all_var:3.1%}")
print(f"Frequency and optimizer var: {1 - sum(freq_optim_vars)/all_var:3.1%}")